In [1]:
# ============================================================
# PRISM (Cendrowska – Canonical)
# ------------------------------------------------------------
# Υλοποίηση του αλγορίθμου PRISM για επαγωγή κανόνων
# IF–THEN από ΚΑΤΗΓΟΡΙΚΑ δεδομένα.
#
# Χαρακτηριστικά:
# - Εκμάθηση κανόνων ξεχωριστά για κάθε κλάση
# - Separate-and-conquer προσέγγιση
# - Σωστό tie-breaking κατά την επιλογή όρων
# - Default class (πλειοψηφική κλάση)
# - Prediction σε νέα δεδομένα
# ============================================================

import pandas as pd
from pandas.api.types import is_numeric_dtype


class CanonicalPrism:
    """
    Κλάση υλοποίησης του Canonical PRISM.

    Ο αλγόριθμος:
    - Μαθαίνει κανόνες της μορφής IF (conditions) THEN class
    - Δημιουργεί κανόνες ξεχωριστά για κάθε τιμή της κλάσης-στόχου
    - Αφαιρεί τα παραδείγματα που καλύπτονται από κάθε κανόνα
    """

    def __init__(self):
        # Λεξικό κανόνων:
        # key   = κλάση-στόχος
        # value = λίστα κανόνων (κάθε κανόνας είναι λίστα από (attribute, value))
        self.rules = {}

        # Όνομα στήλης στόχου
        self.target_col = None

        # Default class:
        # χρησιμοποιείται όταν κανένας κανόνας δεν ταιριάζει
        self.default_target = None

    # --------------------------------------------------------
    # Επιλογή καλύτερου όρου (attribute = value)
    # --------------------------------------------------------
    # Για κάθε attribute και κάθε δυνατή τιμή του:
    # Υπολογίζουμε:
    #   P(class | attribute = value)
    #
    # Tie-breaking:
    # 1. Μέγιστη πιθανότητα
    # 2. Περισσότερα θετικά παραδείγματα
    # 3. Μεγαλύτερη συνολική κάλυψη
    # --------------------------------------------------------
    def _best_term(self, df, target_col, target_val, used_cols):

        best_term = None
        best_prob = -1
        best_pos_covered = -1
        best_total_covered = -1

        # Εξετάζουμε όλα τα attributes εκτός από:
        # - τη στήλη στόχου
        # - attributes που έχουν ήδη χρησιμοποιηθεί στον κανόνα
        for col in df.columns:
            if col == target_col or col in used_cols:
                continue

            # Για κάθε πιθανή τιμή του attribute
            for val in df[col].unique():

                # Υποσύνολο δεδομένων που ικανοποιεί τον όρο
                subset = df[df[col] == val]

                if len(subset) == 0:
                    continue

                # Πλήθος θετικών παραδειγμάτων
                pos = len(subset[subset[target_col] == target_val])

                # Συνολικό πλήθος παραδειγμάτων
                total = len(subset)

                # Υπό συνθήκη πιθανότητα
                prob = pos / total

                # Εφαρμογή κριτηρίων επιλογής
                if prob > best_prob:
                    best_prob = prob
                    best_term = (col, val)
                    best_pos_covered = pos
                    best_total_covered = total

                elif prob == best_prob:
                    if pos > best_pos_covered:
                        best_term = (col, val)
                        best_pos_covered = pos
                        best_total_covered = total
                    elif pos == best_pos_covered and total > best_total_covered:
                        best_term = (col, val)
                        best_total_covered = total

        return best_term, best_prob

    # --------------------------------------------------------
    # Εκμάθηση κανόνων για μία συγκεκριμένη κλάση
    # --------------------------------------------------------
    def _learn_rules_for_class(self, df, target_col, target_val):

        # Αντίγραφο δεδομένων που "μικραίνει" καθώς αφαιρούμε
        # παραδείγματα που καλύπτονται από κανόνες
        df_remaining = df.copy()
        class_rules = []

        # Συνεχίζουμε όσο υπάρχουν παραδείγματα της κλάσης-στόχου
        while (df_remaining[target_col] == target_val).any():

            rule = []          # Ο υπό κατασκευή κανόνας
            used_cols = []     # Attributes που έχουν ήδη χρησιμοποιηθεί
            df_work = df_remaining.copy()

            # Προσθήκη όρων μέχρι ο κανόνας να γίνει "καθαρός"
            while True:
                term, prob = self._best_term(
                    df_work, target_col, target_val, used_cols
                )

                if term is None:
                    break

                col, val = term
                rule.append((col, val))
                used_cols.append(col)

                # Περιορίζουμε το σύνολο δεδομένων
                df_work = df_work[df_work[col] == val]

                # Αν όλα τα παραδείγματα ανήκουν στην κλάση → stop
                if all(df_work[target_col] == target_val):
                    break

            # Αποθήκευση κανόνα
            class_rules.append(rule)

            # Αφαίρεση παραδειγμάτων που καλύφθηκαν
            df_remaining = df_remaining.drop(df_work.index)

        return class_rules

    # --------------------------------------------------------
    # Εκπαίδευση του μοντέλου PRISM
    # --------------------------------------------------------
    def fit(self, df, target_col):

        # Έλεγχος: όλα τα χαρακτηριστικά πρέπει να είναι κατηγορικά
        for col in df.columns:
            if is_numeric_dtype(df[col]):
                raise ValueError(
                    f"Ο PRISM απαιτεί κατηγορικά χαρακτηριστικά. "
                    f"Η στήλη '{col}' είναι αριθμητική."
                )

        self.target_col = target_col
        self.rules = {}

        # Εκμάθηση κανόνων ξεχωριστά για κάθε κλάση
        for target_val in df[target_col].unique():
            self.rules[target_val] = self._learn_rules_for_class(
                df, target_col, target_val
            )

        # Default class = πλειοψηφική κλάση
        self.default_target = df[target_col].mode()[0]

        return self.rules



# ============================================================
# ΠΑΡΑΔΕΙΓΜΑ ΧΡΗΣΗΣ – PlayTennis
# ============================================================

# Εκπαιδευτικό σύνολο δεδομένων
data = {
    "Outlook": ["Sunny", "Sunny", "Overcast", "Rain", "Rain",
                "Overcast", "Sunny", "Rain", "Overcast"],
    "Humidity": ["High", "High", "High", "Normal", "Normal",
                 "Normal", "Normal", "High", "High"],
    "Wind": ["Weak", "Strong", "Weak", "Weak", "Strong",
             "Strong", "Weak", "Weak", "Strong"],
    "PlayTennis": ["No", "No", "Yes", "Yes", "No",
                   "Yes", "Yes", "No", "Yes"]
}

df = pd.DataFrame(data)

# Εκπαίδευση μοντέλου
prism = CanonicalPrism()
rules = prism.fit(df, target_col="PlayTennis")

# Εκτύπωση κανόνων
for cls, rs in rules.items():
    print(f"\nCLASS = {cls}")
    for i, r in enumerate(rs, 1):
        conditions = " AND ".join(f"{c} = {v}" for c, v in r)
        print(f"Rule {i}: IF {conditions} THEN {cls}")

print("\nDefault class:", prism.default_target)





import xml.etree.ElementTree as ET
from xml.dom import minidom
import uuid


# =========================
# ΕΞΑΓΩΓΗ ΚΑΝΟΝΩΝ PRISM ΣΕ CAMUNDA DMN
# =========================

def export_prism_to_dmn(prism, decision_name, filename):
    """
    Μετατρέπει τους PRISM κανόνες (CanonicalPrism)
    σε DMN decision table συμβατό με Camunda
    """

    # ---------------------------------
    # Όλα τα input attributes που εμφανίζονται σε κανόνες
    # ---------------------------------
    inputs = sorted({
        term[0]
        for rules in prism.rules.values()
        for rule in rules
        for term in rule
    })

    # ---------------------------------
    # Ρίζα DMN
    # ---------------------------------
    definitions = ET.Element(
        "definitions",
        {
            "xmlns": "https://www.omg.org/spec/DMN/20191111/MODEL/",
            "xmlns:camunda": "http://camunda.org/schema/1.0/dmn",
            "id": f"def_{uuid.uuid4()}",
            "name": decision_name,
            "namespace": "http://camunda.org/schema/1.0/dmn"
        }
    )

    # Απόφαση
    decision = ET.SubElement(
        definitions,
        "decision",
        {
            "id": f"dec_{uuid.uuid4()}",
            "name": decision_name
        }
    )

    # Decision Table
    table = ET.SubElement(
        decision,
        "decisionTable",
        {
            "id": f"dt_{uuid.uuid4()}",
            "hitPolicy": "FIRST"
        }
    )

    # ---------------------------------
    # Inputs (στήλες πίνακα)
    # ---------------------------------
    for inp in inputs:
        i = ET.SubElement(table, "input")
        expr = ET.SubElement(i, "inputExpression", {"typeRef": "string"})
        ET.SubElement(expr, "text").text = inp

    # Output
    ET.SubElement(
        table,
        "output",
        {
            "name": prism.target_col,
            "typeRef": "string"
        }
    )

    # ---------------------------------
    # Κανόνες → Γραμμές πίνακα
    # ---------------------------------
    for class_label, rules in prism.rules.items():

        for rule in rules:
            r = ET.SubElement(table, "rule")

            rule_map = {attr: val for attr, val in rule}

            for inp in inputs:
                e = ET.SubElement(r, "inputEntry")
                ET.SubElement(e, "text").text = (
                    f"\"{rule_map[inp]}\"" if inp in rule_map else "-"
                )

            out = ET.SubElement(r, "outputEntry")
            ET.SubElement(out, "text").text = str(class_label)

    # ---------------------------------
    # Pretty XML
    # ---------------------------------
    xml = minidom.parseString(
        ET.tostring(definitions, encoding="utf-8")
    ).toprettyxml(indent="  ")

    with open(filename, "w", encoding="utf-8") as f:
        f.write(xml)

    print(f"\nDMN αρχείο δημιουργήθηκε: {filename}")


# =========================
# ΔΗΜΙΟΥΡΓΙΑ DMN ΑΡΧΕΙΟΥ
# =========================

export_prism_to_dmn(
    prism,
    decision_name="PRISM Decision Table",
    filename="prism_decision_table.dmn"
)





CLASS = No
Rule 1: IF Outlook = Sunny AND Humidity = High THEN No
Rule 2: IF Outlook = Rain AND Humidity = High THEN No
Rule 3: IF Outlook = Rain AND Wind = Strong THEN No

CLASS = Yes
Rule 1: IF Outlook = Overcast THEN Yes
Rule 2: IF Humidity = Normal AND Wind = Weak THEN Yes

Default class: Yes

DMN αρχείο δημιουργήθηκε: prism_decision_table.dmn
